# Voiceoftaade — MoneyPrinterTurbo Colab Launcher

This notebook launches the MoneyPrinterTurbo Streamlit dashboard in an existing Google Colab runtime and exposes it through an email-protected Cloudflare Quick Tunnel.

## Before you run it
- Run the installation/setup cell from the official MoneyPrinterTurbo Colab notebook first.
- This launcher expects the project at `/content/MoneyPrinterTurbo` and Streamlit on port `8501`.
- When prompted, enter the email address that should be allowed to access the dashboard. Cloudflare sends that address a one-time PIN.
- Do not share the tunnel URL. Stop the tunnel when you finish.
- Colab runtimes are temporary; the tunnel URL changes when restarted.

The tunnel is for temporary testing, not production hosting. This notebook does not disable Streamlit's CORS or XSRF protections and does not use ngrok.

## 1. Install MoneyPrinterTurbo

Run this cell first. It clones the official repository and installs its locked dependencies in an isolated Python 3.11 environment. Wait for `INSTALLATION COMPLETE` before moving on.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/harry0703/MoneyPrinterTurbo.git"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
elif REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} exists but is not a Git repository. "
        "Rename or remove that folder before continuing."
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
subprocess.run(
    ["python", "-m", "pip", "install", "-q", "uv"],
    check=True,
)
subprocess.run(
    ["uv", "python", "install", "3.11"],
    check=True,
)
subprocess.run(
    ["uv", "sync", "--frozen", "--python", "3.11"],
    check=True,
)

print("INSTALLATION COMPLETE")
print("Project folder:", REPO_DIR)
print("Next: run the dashboard launch cell.")


## 2. Launch the dashboard

Run this only after installation completes. It starts Streamlit and displays port 8501 using Google Colab's built-in port display. No ngrok token or third-party public tunnel is used.


In [ ]:
import subprocess
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen
from IPython.display import display
from google.colab import output

REPO_DIR = Path("/content/MoneyPrinterTurbo")
PORT = 8501
LOG_PATH = Path("/content/moneyprinterturbo-webui.log")

if not (REPO_DIR / "webui" / "Main.py").is_file():
    raise RuntimeError(
        "MoneyPrinterTurbo is not installed. "
        "Run the cell named '1. Install MoneyPrinterTurbo' first."
    )

def app_is_healthy():
    try:
        with urlopen(
            f"http://127.0.0.1:{PORT}/_stcore/health",
            timeout=3,
        ) as response:
            return response.status == 200
    except (URLError, TimeoutError, OSError):
        return False

# Reuse an already-running healthy app.
if app_is_healthy():
    print("MoneyPrinterTurbo is already running.")
else:
    log_handle = LOG_PATH.open("w", encoding="utf-8")
    streamlit_proc = subprocess.Popen(
        [
            "uv", "run", "streamlit", "run", "webui/Main.py",
            f"--server.port={PORT}",
            "--server.address=0.0.0.0",
            "--browser.gatherUsageStats=False",
            "--client.toolbarMode=minimal",
            "--server.showEmailPrompt=False",
        ],
        cwd=REPO_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

    deadline = time.time() + 90
    while time.time() < deadline:
        if app_is_healthy():
            break
        if streamlit_proc.poll() is not None:
            break
        time.sleep(2)

    if not app_is_healthy():
        log_handle.flush()
        recent_log = LOG_PATH.read_text(
            encoding="utf-8",
            errors="replace",
        )[-4000:]
        raise RuntimeError(
            "Streamlit did not start. Recent log:\n" + recent_log
        )

print("Dashboard is healthy on port", PORT)
print("Opening it through Google Colab's built-in port display.")
output.serve_kernel_port_as_iframe(PORT, height=900)


## Stop the dashboard

Run this when finished to stop the Streamlit process started by this notebook.


In [ ]:
proc = globals().get('streamlit_proc')
if proc is not None and proc.poll() is None:
    proc.terminate()
    proc.wait(timeout=10)
    print('Dashboard stopped.')
else:
    print('No active dashboard process found.')
